In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [ ]:
# Environment setup
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scrublet as scr
import scipy.stats as stats

In [ ]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "scRNA_Glyco/sc_myeloid"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "scRNA_Glyco/sc_myeloid/figure"
sc.settings.set_figure_params(dpi=200, format="pdf", dpi_save=800) # set sufficiently high resolution for saving 400dpi

In [ ]:
###--- Load pre-filtering data ---###

In [ ]:
# Data Loading
adata_celltypist = sc.read("sc_celltypist_myeloid.h5ad")
# Start with Raw data
adata_celltypist.X = adata_celltypist.layers["counts"].copy()

In [ ]:
adata_celltypist

In [ ]:
###--- Normalization, Highly Variable Gene, scale ---###

In [ ]:
# Normalization
sc.pp.normalize_total(adata_celltypist, target_sum=1e4)
adata_celltypist.layers["norm10k"] = adata_celltypist.X

In [ ]:
# Logarithmize the data:
sc.pp.log1p(adata_celltypist)
adata_celltypist.layers["log1p"] = adata_celltypist.X

In [ ]:
# Highly Variable Gene Selection
sc.pp.highly_variable_genes(adata_celltypist, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata_celltypist)

In [ ]:
#Freeze the state of the AnnData object
adata_celltypist.raw = adata_celltypist
adata_celltypist.raw.var.index.is_unique

In [ ]:
# Scale each gene to unit variance.
sc.pp.scale(adata_celltypist, max_value=10)

In [ ]:
###--- Integration with Harmony ---###

In [ ]:
#Data visualization: Clustering

In [ ]:
#Dimensionality Reduction
sc.pp.pca(adata_celltypist, svd_solver="arpack", use_highly_variable=True,  n_comps=60)
sc.pl.pca_variance_ratio(adata_celltypist, log=True, n_pcs=50)

In [ ]:
#Computing the neighborhood graph
sc.pp.neighbors(adata_celltypist, n_neighbors=15, n_pcs=40)
sc.tl.umap(adata_celltypist)

In [ ]:
sc.tl.leiden(adata_celltypist, key_added="leiden", resolution=1.1)

In [ ]:
sc.pl.umap(adata_celltypist, color=['leiden','donor', 'label', 'sample'],   wspace=0.5, save='_batch_before_harmony_mye')

In [ ]:
#Running Harmony
sce.pp.harmony_integrate(adata_celltypist, 'donor')

In [ ]:
adata_celltypist.obsm['X_pca'] = adata_celltypist.obsm['X_pca_harmony']

In [ ]:
sc.pp.neighbors(adata_celltypist, n_neighbors=20, n_pcs = 50)
sc.tl.umap(adata_celltypist)

In [ ]:
sc.tl.leiden(adata_celltypist, resolution=1)

In [ ]:
sc.pl.umap(adata_celltypist, color=['leiden','donor', 'label', 'sample'],   wspace=0.5, save='_batch_after_harmony_mye')

In [ ]:
# cell_type counts
cell_type_counts = adata_celltypist.obs['leiden'].value_counts()
print(cell_type_counts)

In [ ]:
# Set umap palette colors
umap_cluster_col = ['#c3e0ea', '#5593BE', '#83D4F2','#8CAED8', '#B7D0F4', '#6F8ACF', '#DAD9EA', '#A5A7C5', '#8F8DB8', '#7660A0', '#AB87B8', '#8D7590', '#AC95A9', '#C667B3', '#DAB6D0', '#E594C6', '#F0B9C6', '#EB778F', '#E1888C', '#B6B6B6']
leiden_colors = {str(i): umap_cluster_col[i] for i in range(20)}

In [ ]:
sc.pl.umap(adata_celltypist, color=['leiden'], save='_marker_cluster_mye', legend_loc='on data',  palette=leiden_colors)

In [ ]:
###--- IDENTIFYING CELLULAR STRUCTURE ---###

In [ ]:
#- Differentially expressed genes: TOP MARKER -#
sc.tl.rank_genes_groups(adata_celltypist, groupby='leiden', reference='rest', method='wilcoxon' , key_added="dea_leiden", pts=True)

top_markers = pd.DataFrame(adata_celltypist.uns['dea_leiden']['names']).head(100)
print(top_markers)

In [ ]:
# save results
result = adata_celltypist.uns['dea_leiden']
groups = result['names'].dtype.names
result_df = pd.DataFrame(
    {group + '_' + key[:10]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals','pvals_adj','pts','pts_rest']}).head(100)
result_df
result_df.to_csv("differential_expression_results_All_myeloid_clusters.csv", index=False)

In [ ]:
#--- TUMOR-ASSOCIATED MACROPHAGES ---#
adata_TAMs = adata_celltypist[adata_celltypist.obs['leiden'].isin(['0','1','5','6','4','3','8', '14'])].copy()
adata_TAMs

In [ ]:
#- Differentially expressed genes -#
sc.tl.rank_genes_groups(adata_TAMs, groupby='leiden', reference='rest', method='wilcoxon' , key_added="dea_leiden_TAMs", pts=True)

In [ ]:
result = adata_TAMs.uns['dea_leiden_TAMs']
groups = result['names'].dtype.names
result_df_TAM = pd.DataFrame(
    {group + '_' + key[:10]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals','pvals_adj','pts','pts_rest']}).head(100)

result_df_TAM.to_csv("differential_expression_results_TAMs.csv", index=False)

In [ ]:
# Filtering: min_in_group_fraction: 0.25 min_fold_change: 1, max_out_group_fraction: 0.5
sc.tl.filter_rank_genes_groups(adata_TAMs, min_fold_change=1, key='dea_leiden_TAMs')
adata_TAMs.uns['rank_genes_groups_filtered']
top_markers = pd.DataFrame(adata_TAMs.uns['rank_genes_groups_filtered']['names']).head(50)
print(top_markers)

top_markers.to_csv("differential_expression_results_top_markersTAMs.csv", index=False)

In [ ]:
###--- Annotation ---###

In [ ]:
# Specific subfamily Marker genes visualization

In [ ]:
sc.pl.umap(adata_celltypist, color=['MKI67', 'KIT' , 'GATA2' , 'CCR3' , 'HPGDS', 'CSF3R' , 'ALDH1A2' , 'NRXN1' ,  'ARG1',  'IRF8' , 'CLEC4C', 'TLR7',  'CLEC9A' , 'CADM1' ], cmap='viridis',  save='_marker_myeloidi_global')

In [ ]:
#Specific markert to TAM subsets
sc.pl.umap(adata_celltypist, color=['IRF1','IRF7','MRC1' ,  'C1QA' , 'C1QB', 'AOAH' , 'HLA-DRA' , 'HLA-DPA1' , 'HLA-DQA1', 'VCAN', 'FCN1' ,  'PPARG' , 'THBS1' , 
                                     'IL1B' , 'NLRP3' , 'CXCL8' , 'MKI67', 'TOP2A'], cmap='viridis',  save='_marker_myeloidi_TAMs')

In [ ]:
#Myeloid UMAP MARKER GENES: 
marker_genes_dict_my = {
'Proliferative' : [ 'MKI67' , 'TOP2A','STMN1'],    
'Lineage TAMs' : ['CD68','MRC1'],
'Angiogenic TAMs' : ['VCAN','FCN1' ,  'PPARG' , 'THBS1'],
'IL1B TAMs' : ['IL1B' , 'NLRP3' , 'CXCL8'],
'LA TAMs' : ['C1QA' , 'C1QB'],
'HSP+ TAMs' : ['HSPH1' , 'HSPD1','BAG3'],
'CSF1R+ TAMs' : ['CSF1R'],   
'Neutrophils' : ['CSF3R' , 'ALDH1A2' , 'NRXN1' ,  'ARG1'],
'cDC1' : ['IRF8','CLEC9A' , 'CADM1'],
'pDCs' : ['CLEC4C', 'TLR7'],
'Mast' : ['KIT' , 'GATA2' , 'CCR3' , 'HPGDS']
    
}

In [ ]:
sc.pl.dotplot(adata_celltypist, var_names=marker_genes_dict_my, groupby='leiden', color_map='viridis', vmin=0, vmax=1, standard_scale='var', save='Global_marker_specificgenes_subfamily_annotation_mye.pdf');

In [ ]:
# Save the state of the AnnData object Counts.
adata_pp = adata_celltypist.copy()
adata_celltypist.write("sc_myeloid_clustering.h5ad")
del adata_pp

In [ ]:
###--- Evaluation of cell typy distribution ---###

In [ ]:
adata_celltypist.obs['sample']

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# List of annotation categories
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata_celltypist.obs['sample'].isin(comparison_id)
subset_adata_ly_cmp = adata_celltypist[boolean_mask, :]

# Create the cross-tabulated data
tmp = pd.crosstab( subset_adata_ly_cmp.obs['label'], subset_adata_ly_cmp.obs['leiden'], normalize='index')

# Create the horizontal stacked bar plot
ax = tmp.plot(kind='bar',stacked=True, edgecolor='none',  color=leiden_colors)

# Define labels and legend
horiz_offset = 1.03
vert_offset = 1.
ax.legend(bbox_to_anchor=(horiz_offset, vert_offset))
ax.set_ylabel("Normalized Counts")
ax.set_xlabel("Sample")
ax.set_title("Normalized Stacked Bar Plot of Annotation Categories")

# Save the plot
plt.savefig('scRNA_Glyco/figure/myeloidy_categories_barplot_all_clusters.pdf', bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
#Cell typy distribution TAM subsets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# List of annotation categories
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata_TAMs.obs['sample'].isin(comparison_id)
subset_adata_my_cmp = adata_TAMs[boolean_mask, :]

# Create the cross-tabulated data
tmp = pd.crosstab(subset_adata_my_cmp.obs['label'], subset_adata_my_cmp.obs['leiden'], normalize='index')

# Create the horizontal stacked bar plot
ax = tmp.plot(kind='bar',stacked=True, edgecolor='none', color=leiden_colors)

# Define labels and legend
horiz_offset = 1.03
vert_offset = 1.
ax.legend(bbox_to_anchor=(horiz_offset, vert_offset))
ax.set_ylabel("Normalized Counts")
ax.set_xlabel("Sample")
ax.set_title("Normalized Stacked Bar Plot of Annotation Categories")

# Save the plot
plt.savefig('scRNA_Glyco/figure/myeloidy_categories_barplot_TAMs.pdf', bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
import scanpy as sc

# Select three samples of interest
samples_of_interest = ['CAR','Tr2DG','TrTUN']

# Subset the AnnData object to keep only observations from the selected samples
adata_filtered = adata_celltypist[adata_celltypist.obs['label'].isin(samples_of_interest)]

# Plot UMAPs for each sample, coloring by Leiden clusters
for sample in samples_of_interest:
    sc.pl.umap(adata_filtered[adata_filtered.obs['label'] == sample],wspace=0.5, color='leiden', title=f'UMAP - {sample}')
    plt.savefig('myeloidy_categories_lost_cluster_samples.pdf', bbox_inches='tight')

In [ ]:
#Correlation with gene signatures

In [ ]:
# IL1B+ TAM signature
gene_list_ILbpos = []
gene_list_ILbpos.extend(['CCL3L1', 'IGHG3', 'CCR7', 'IGKC', 'MIR155HG', 'IL1B', 'IL1A', 'IGHG1', 'CCL4L2', 'IL6', 'TNF', 'CCL4', 'CCL20', 'TCHH', 'IGHA1', 'CCL3', 'PDGFB', 'NLRP3', 'FCGBP', 'THAP2',  'CXCL1', 'BIRC3', 'NFKB1',  'LINC00513', 'CXCL3', 'TRAF1', 'ICAM4', 'TCOF1', 'PTGS2', 'TNFAIP2', 'PLEKHG2', 'CXCL2', 'ICAM1', 'TNFAIP3', 'CXCL8', 'SRGAP1', 'IL10', 'NFKBIA', 'DNAAF1', 'GPR84', 'IER3', 'PDE4B', 'RRAD', 'C3', 'KDM6B', 'CFLAR', 'TNFAIP6', 'SERPINB9', 'GPR132', 'CYB5D1', 'IL1R2', 'SLC7A5', 'NFKBIZ', 'EREG', 'CCL5', 'PLAUR', 'GPR183', 'IRAK2', 'EIF4E', 'ZNF267'])
sc.tl.score_genes(adata_celltypist, gene_list_ILbpos, ctrl_size=50, gene_pool=None, n_bins=25, score_name='Ilb1pos', random_state=0, copy=False, use_raw=None)

In [ ]:
import scvelo as scv
rcParams['axes.grid']=False
scv.pl.scatter(adata_celltypist, color=['Ilb1pos'], perc=[2,98], cmap='PuRd', vmin=0 , save='scRNA_Glyco/figure/gene_list_ILbpos_score.pdf')

In [ ]:
sc.pl.umap(adata_celltypist, color=['MRC1', 'IL1B', 'NLRP3', 'CXCL8' ], cmap='viridis', save='_IL1B_markers.pdf')